# Merge Raw Data
Join `entry_info`, `riders_rating`, and `bikes_rating` into a single processed dataset.

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

In [2]:
entry_info    = pd.read_csv(RAW / 'entry_info.csv')
riders_rating = pd.read_csv(RAW / 'riders_rating.csv')
bikes_rating  = pd.read_csv(RAW / 'bikes_rating.csv')

print('entry_info:   ', entry_info.shape)
print('riders_rating:', riders_rating.shape)
print('bikes_rating: ', bikes_rating.shape)

entry_info:    (24, 7)
riders_rating: (24, 7)
bikes_rating:  (12, 7)


## Preview

In [3]:
entry_info.head()

,name,age,nationality,bike_number,manufacturer,team,team_status
0,Javier Ruiz,28,Spain,26,Ducati,Razor Racing,satellite
1,Sebastian Aginaza,23,Spain,88,Ducati,Razor Racing,satellite
2,Davide Greco,31,Italy,77,Triumph,Triumph Factory Racing,factory
3,Gabriel Silva,23,Brazil,64,Triumph,Triumph Factory Racing,factory
4,Dwi Gunawan,21,Indonesia,32,BMW,Falcon Racing,satellite


In [4]:
riders_rating.head()

,name,braking,cornering,aggression,tyre_management,wet_performance,consistency
0,Javier Ruiz,79,83,99,93,93,99
1,Sebastian Aginaza,96,81,91,76,71,75
2,Davide Greco,79,86,70,87,77,99
3,Gabriel Silva,79,84,70,92,93,76
4,Dwi Gunawan,96,84,72,71,86,81


In [5]:
bikes_rating

,manufacturer,team_status,top_speed,acceleration,braking,cornering,stability
0,BMW,satellite,83,83,68,81,78
1,BMW,factory,88,88,73,86,83
2,Ducati,satellite,92,86,86,85,85
3,Ducati,factory,97,91,91,90,90
4,Honda,satellite,80,84,73,80,83
5,Honda,factory,85,89,78,85,88
6,Kawasaki,factory,86,89,89,88,93
7,Suzuki,factory,87,89,85,94,91
8,Triumph,satellite,72,73,74,70,72
9,Triumph,factory,77,78,79,75,77


## Join

- Step 1: `entry_info` ← LEFT JOIN `riders_rating` ON `name`
- Step 2: result ← LEFT JOIN `bikes_rating` ON `manufacturer` + `team_status`
- `braking` and `cornering` exist in both rider and bike tables → renamed with `rider_` / `bike_` prefix

In [6]:
# Rename conflicting columns before joining
riders_rating = riders_rating.rename(columns={
    'braking':  'rider_braking',
    'cornering': 'rider_cornering'
})

bikes_rating = bikes_rating.rename(columns={
    'braking':  'bike_braking',
    'cornering': 'bike_cornering'
})

In [7]:
df = (
    entry_info
    .merge(riders_rating, on='name', how='left')
    .merge(bikes_rating,  on=['manufacturer', 'team_status'], how='left')
)

print('Merged shape:', df.shape)
df.head()

Merged shape: (24, 18)


,name,age,nationality,bike_number,manufacturer,team,team_status,rider_braking,rider_cornering,aggression,tyre_management,wet_performance,consistency,top_speed,acceleration,bike_braking,bike_cornering,stability
0,Javier Ruiz,28,Spain,26,Ducati,Razor Racing,satellite,79,83,99,93,93,99,92,86,86,85,85
1,Sebastian Aginaza,23,Spain,88,Ducati,Razor Racing,satellite,96,81,91,76,71,75,92,86,86,85,85
2,Davide Greco,31,Italy,77,Triumph,Triumph Factory Racing,factory,79,86,70,87,77,99,77,78,79,75,77
3,Gabriel Silva,23,Brazil,64,Triumph,Triumph Factory Racing,factory,79,84,70,92,93,76,77,78,79,75,77
4,Dwi Gunawan,21,Indonesia,32,BMW,Falcon Racing,satellite,96,84,72,71,86,81,83,83,68,81,78


In [8]:
print('Null values:\n', df.isnull().sum())

Null values:
 name               0
age                0
nationality        0
bike_number        0
manufacturer       0
team               0
team_status        0
rider_braking      0
rider_cornering    0
aggression         0
tyre_management    0
wet_performance    0
consistency        0
top_speed          0
acceleration       0
bike_braking       0
bike_cornering     0
stability          0
dtype: int64


## Export

In [9]:
out_path = PROCESSED / 'riders_full.csv'
df.to_csv(out_path, index=False)
print(f'Saved to {out_path}  ({len(df)} rows, {len(df.columns)} columns)')
print('Columns:', list(df.columns))

Saved to ..\data\processed\riders_full.csv  (24 rows, 18 columns)
Columns: ['name', 'age', 'nationality', 'bike_number', 'manufacturer', 'team', 'team_status', 'rider_braking', 'rider_cornering', 'aggression', 'tyre_management', 'wet_performance', 'consistency', 'top_speed', 'acceleration', 'bike_braking', 'bike_cornering', 'stability']
